<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/housing_mcda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import minmax_scale
import warnings
warnings.filterwarnings('ignore')

# For advanced visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Decision hierarchy based on your tree
criteria_hierarchy = {
    'Employment': {
        'weight': 0.25,
        'sub_criteria': {
            'HR Data': {
                'weight': 0.40,
                'sub_criteria': {
                    'Hybrid Policy': {'weight': 0.50, 'direction': 'max'},
                    'Job Availability': {'weight': 0.50, 'direction': 'max'}
                }
            },
            'Salary Potential': {'weight': 0.60, 'direction': 'max'}
        }
    },
    'Accommodation': {
        'weight': 0.25,
        'sub_criteria': {
            'Rent/Deposit': {
                'weight': 0.70,
                'sub_criteria': {
                    'Monthly Rent': {'weight': 0.60, 'direction': 'min'},
                    'Deposit Required': {'weight': 0.40, 'direction': 'min'}
                }
            },
            'Availability': {'weight': 0.30, 'direction': 'max'}
        }
    },
    'Transport': {
        'weight': 0.15,
        'sub_criteria': {
            'Commute Time': {'weight': 0.50, 'direction': 'min'},
            'Reliability': {'weight': 0.30, 'direction': 'max'},
            'Public Transport': {'weight': 0.20, 'direction': 'max'}
        }
    },
    'Financial Costs': {
        'weight': 0.15,
        'sub_criteria': {
            'Salary': {'weight': 0.40, 'direction': 'max'},
            'Budget': {'weight': 0.35, 'direction': 'min'},
            'Savings': {'weight': 0.25, 'direction': 'max'}
        }
    },
    'Personal Preferences': {
        'weight': 0.12,
        'sub_criteria': {
            'Wellbeing': {'weight': 0.40, 'direction': 'max'},
            'Lifestyle': {'weight': 0.35, 'direction': 'max'},
            'Flexibility': {'weight': 0.25, 'direction': 'max'}
        }
    },
    'External Risks': {
        'weight': 0.08,
        'sub_criteria': {
            'Weather': {'weight': 0.20, 'direction': 'max'},
            'Crime': {'weight': 0.50, 'direction': 'min'},
            'Flooding': {'weight': 0.30, 'direction': 'min'}
        }
    }
}

In [ ]:
# Raw scores for each location (1-10 scale, 10 = best for that criterion)
data = {
    'Criterion': [
        'Hybrid Policy', 'Job Availability', 'Salary Potential',
        'Monthly Rent', 'Deposit Required', 'Availability',
        'Commute Time', 'Reliability', 'Public Transport',
        'Net Income', 'Living Costs', 'Savings Potential',
        'Quality of Life', 'Social/Cultural', 'Remote Work Flex',
        'Climate', 'Safety', 'Flooding Risk'
    ],
    'Direction': [
        'max', 'max', 'max',
        'min', 'min', 'max',
        'min', 'max', 'max',
        'max', 'min', 'max',
        'max', 'max', 'max',
        'max', 'min', 'min'
    ],
    'London': [9, 10, 10, 3, 4, 10, 5, 8, 10, 10, 2, 9, 8, 10, 8, 5, 6, 7],
    'Manchester': [7, 8, 6, 8, 7, 8, 8, 7, 7, 5, 8, 4, 6, 7, 9, 4, 7, 5],
    'Bristol': [8, 6, 7, 7, 7, 6, 7, 6, 6, 6, 7, 5, 9, 8, 8, 7, 8, 4],
    'Edinburgh': [6, 5, 5, 6, 6, 4, 6, 8, 8, 4, 6, 3, 10, 9, 7, 3, 10, 9],
    'Birmingham': [5, 7, 6, 9, 8, 9, 8, 5, 5, 5, 9, 4, 4, 5, 6, 5, 5, 6],
    'Cambridge': [9, 4, 8, 4, 5, 3, 9, 9, 4, 7, 3, 6, 7, 4, 10, 6, 9, 8]
}

# Create DataFrame
df = pd.DataFrame(data)
df.set_index('Criterion', inplace=True)
print("Raw Data:")
print(df.round(1))

Raw Data:
                  Direction  London  Manchester  Bristol  Edinburgh  \
Criterion                                                             
Hybrid Policy           max       9           7        8          6   
Job Availability        max      10           8        6          5   
Salary Potential        max      10           6        7          5   
Monthly Rent            min       3           8        7          6   
Deposit Required        min       4           7        7          6   
Availability            max      10           8        6          4   
Commute Time            min       5           8        7          6   
Reliability             max       8           7        6          8   
Public Transport        max      10           7        6          8   
Net Income              max      10           5        6          4   
Living Costs            min       2           8        7          6   
Savings Potential       max       9           4        5          3

In [ ]:
def calculate_weights(hierarchy):
    """Calculate final weights for each leaf criterion"""
    weights = {}

    def traverse(node, parent_weight=1.0, path=[]):
        if 'sub_criteria' in node:
            # Distribute parent weight among sub-criteria
            for name, subnode in node['sub_criteria'].items():
                new_path = path + [name]
                if 'sub_criteria' in subnode:
                    traverse(subnode, parent_weight * node['weight'], new_path)
                else:
                    # Leaf node
                    full_name = ' > '.join(new_path)
                    weights[full_name] = {
                        'weight': parent_weight * subnode['weight'],
                        'direction': subnode['direction']
                    }
        else:
            # Leaf node
            full_name = ' > '.join(path + [name])
            weights[full_name] = {
                'weight': parent_weight * node['weight'],
                'direction': node['direction']
            }

    # Start traversal from root
    for name, node in hierarchy.items():
        if 'sub_criteria' in node:
            for subname, subnode in node['sub_criteria'].items():
                if 'sub_criteria' in subnode:
                    traverse(subnode, node['weight'], [name, subname])
                else:
                    full_name = f"{name} > {subname}"
                    weights[full_name] = {
                        'weight': node['weight'] * subnode['weight'],
                        'direction': subnode['direction']
                    }
        else:
            weights[name] = {
                'weight': node['weight'],
                'direction': node['direction']
            }

    return weights

# Calculate weights
criterion_weights = calculate_weights(criteria_hierarchy)

# Create weight DataFrame
weight_df = pd.DataFrame([
    {'Criterion': k, 'Weight': v['weight'], 'Direction': v['direction']}
    for k, v in criterion_weights.items()
])
print("\nCalculated Weights:")
print(weight_df.round(3))


Calculated Weights:
                                          Criterion  Weight Direction
0              Employment > HR Data > Hybrid Policy   0.125       max
1           Employment > HR Data > Job Availability   0.125       max
2                     Employment > Salary Potential   0.150       max
3       Accommodation > Rent/Deposit > Monthly Rent   0.150       min
4   Accommodation > Rent/Deposit > Deposit Required   0.100       min
5                      Accommodation > Availability   0.075       max
6                          Transport > Commute Time   0.075       min
7                           Transport > Reliability   0.045       max
8                      Transport > Public Transport   0.030       max
9                          Financial Costs > Salary   0.060       max
10                         Financial Costs > Budget   0.052       min
11                        Financial Costs > Savings   0.038       max
12                 Personal Preferences > Wellbeing   0.048       max

In [ ]:
def normalize_minmax(df, directions):
    """Normalize using Min-Max method"""
    normalized = df.copy()
    for col in df.columns:
        if col in directions:
            if directions[col] == 'max':
                normalized[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
            else:  # minimize
                normalized[col] = (df[col].max() - df[col]) / (df[col].max() - df[col].min())
    return normalized

def normalize_vector(df, directions):
    """Normalize using Vector method (for TOPSIS)"""
    normalized = df.copy()
    for col in df.columns:
        norm_factor = np.sqrt((df[col]**2).sum())
        if norm_factor > 0:
            normalized[col] = df[col] / norm_factor
    return normalized

def normalize_linear(df, directions):
    """Normalize using Linear method"""
    normalized = df.copy()
    for col in df.columns:
        if directions[col] == 'max':
            normalized[col] = df[col] / df[col].max()
        else:  # minimize
            normalized[col] = df[col].min() / df[col]
    return normalized

# Prepare data for normalization
scores_df = df.drop('Direction', axis=1).T
directions = df['Direction'].to_dict()

print("\nScores Matrix:")
print(scores_df)


Scores Matrix:
Criterion   Hybrid Policy  Job Availability  Salary Potential  Monthly Rent  \
London                  9                10                10             3   
Manchester              7                 8                 6             8   
Bristol                 8                 6                 7             7   
Edinburgh               6                 5                 5             6   
Birmingham              5                 7                 6             9   
Cambridge               9                 4                 8             4   

Criterion   Deposit Required  Availability  Commute Time  Reliability  \
London                     4            10             5            8   
Manchester                 7             8             8            7   
Bristol                    7             6             7            6   
Edinburgh                  6             4             6            8   
Birmingham                 8             9             8         

In [ ]:
class WeightedSumModel:
    def __init__(self, scores, weights, directions):
        self.scores = scores
        self.weights = weights
        self.directions = directions

    def calculate(self, normalization='minmax'):
        """Calculate weighted sum scores"""
        # Normalize
        if normalization == 'minmax':
            normalized = normalize_minmax(self.scores, self.directions)
        elif normalization == 'vector':
            normalized = normalize_vector(self.scores, self.directions)
        elif normalization == 'linear':
            normalized = normalize_linear(self.scores, self.directions)
        else:
            normalized = self.scores.copy()

        # Apply weights
        weighted = normalized.copy()
        for col in weighted.columns:
            weighted[col] = weighted[col] * self.weights.get(col, 0)

        # Calculate total score
        weighted['Total_Score'] = weighted.sum(axis=1)

        return weighted.sort_values('Total_Score', ascending=False)

# Create weight mapping (original, hierarchical)
weight_map = dict(zip(weight_df['Criterion'], weight_df['Weight']))
direction_map = dict(zip(weight_df['Criterion'], weight_df['Direction']))

# Establish a mapping from simple data names (scores_df columns) to hierarchical weight names
# This is crucial due to the naming discrepancy between 'data' and 'criteria_hierarchy'
data_to_hierarchy_map = {
    'Hybrid Policy': 'Employment > HR Data > Hybrid Policy',
    'Job Availability': 'Employment > HR Data > Job Availability',
    'Salary Potential': 'Employment > Salary Potential',
    'Monthly Rent': 'Accommodation > Rent/Deposit > Monthly Rent',
    'Deposit Required': 'Accommodation > Rent/Deposit > Deposit Required',
    'Availability': 'Accommodation > Availability',
    'Commute Time': 'Transport > Commute Time',
    'Reliability': 'Transport > Reliability',
    'Public Transport': 'Transport > Public Transport',
    'Net Income': 'Financial Costs > Salary',  # Mapped 'Net Income' to 'Salary'
    'Living Costs': 'Financial Costs > Budget',  # Mapped 'Living Costs' to 'Budget'
    'Savings Potential': 'Financial Costs > Savings', # Mapped 'Savings Potential' to 'Savings'
    'Quality of Life': 'Personal Preferences > Wellbeing', # Mapped 'Quality of Life' to 'Wellbeing'
    'Social/Cultural': 'Personal Preferences > Lifestyle', # Mapped 'Social/Cultural' to 'Lifestyle'
    'Remote Work Flex': 'Personal Preferences > Flexibility', # Mapped 'Remote Work Flex' to 'Flexibility'
    'Climate': 'External Risks > Weather',  # Mapped 'Climate' to 'Weather'
    'Safety': 'External Risks > Crime',    # Mapped 'Safety' to 'Crime'
    'Flooding Risk': 'External Risks > Flooding'  # Mapped 'Flooding Risk' to 'Flooding'
}

# Create new weight and direction maps that align with scores_df columns
aligned_weights = {col: weight_map.get(data_to_hierarchy_map.get(col, col), 0) for col in scores_df.columns}
aligned_directions = directions # The `directions` variable (from cell H7f6XBdV4z5v) already has simple names

# Run WSM
wsm = WeightedSumModel(scores_df, aligned_weights, aligned_directions)
wsm_results = wsm.calculate(normalization='minmax')

print("\n" + "="*60)
print("WEIGHTED SUM MODEL RESULTS")
print("="*60)
print(wsm_results[['Total_Score']].round(4))


WEIGHTED SUM MODEL RESULTS
Criterion   Total_Score
London           1.1524
Cambridge        0.6326
Bristol          0.5478
Manchester       0.4661
Edinburgh        0.4108
Birmingham       0.2662


In [ ]:
class TOPSIS:
    def __init__(self, scores, weights, directions):
        self.scores = scores
        self.weights = weights
        self.directions = directions

    def calculate(self):
        """Calculate TOPSIS scores and rankings"""
        # Step 1: Vector normalization
        normalized = normalize_vector(self.scores, self.directions)

        # Step 2: Weighted normalized matrix
        weighted = normalized.copy()
        for col in weighted.columns:
            # Ensure the criterion exists in the weight map, otherwise use 0 weight
            weighted[col] = weighted[col] * self.weights.get(col, 0)

        # Step 3: Identify ideal and negative-ideal solutions
        ideal = []
        negative_ideal = []

        # Re-calculating ideal and negative ideal using Series for correct alignment
        ideal_series = pd.Series(index=weighted.columns)
        negative_ideal_series = pd.Series(index=weighted.columns)

        for col in weighted.columns:
            if self.directions.get(col) == 'max':
                ideal_series[col] = weighted[col].max()
                negative_ideal_series[col] = weighted[col].min()
            elif self.directions.get(col) == 'min':
                ideal_series[col] = weighted[col].min()
                negative_ideal_series[col] = weighted[col].max()
            else:
                # Handle cases where direction might be missing or not 'max'/'min'
                # For now, if no direction, it means it's not a criterion being optimized
                # This case should ideally not be reached if all columns have directions.
                ideal_series[col] = 0 # Default to 0 or handle as appropriate for the problem
                negative_ideal_series[col] = 0

        # Step 4: Calculate separation measures
        S_plus = np.sqrt(((weighted - ideal_series)**2).sum(axis=1))
        S_minus = np.sqrt(((weighted - negative_ideal_series)**2).sum(axis=1))

        # Step 5: Calculate relative closeness
        C_i = S_minus / (S_plus + S_minus + 1e-10)  # Add small epsilon to avoid division by zero

        # Create results DataFrame
        results = weighted.copy()
        results['S_plus'] = S_plus
        results['S_minus'] = S_minus
        results['Closeness'] = C_i
        results['Rank'] = C_i.rank(ascending=False)

        return results.sort_values('Closeness', ascending=False)

# Run TOPSIS
topsis = TOPSIS(scores_df, aligned_weights, aligned_directions)
topsis_results = topsis.calculate()

print("\n" + "="*60)
print("TOPSIS RESULTS")
print("="*60)
print(topsis_results[['Closeness', 'Rank']].round(4))


TOPSIS RESULTS
Criterion   Closeness  Rank
London         0.9264   1.0
Cambridge      0.5162   2.0
Bristol        0.4029   3.0
Manchester     0.3736   4.0
Edinburgh      0.3473   5.0
Birmingham     0.2938   6.0


In [ ]:
class PROMETHEE:
    def __init__(self, scores, weights, directions):
        self.scores = scores
        self.weights = weights
        self.directions = directions
        self.alternatives = scores.index.tolist()

    def preference_function(self, diff, p=0.5, q=0.1):
        """Gaussian preference function"""
        if diff <= 0:
            return 0
        return 1 - np.exp(-(diff**2) / (2 * p**2))

    def calculate(self):
        """Calculate PROMETHEE scores"""
        n = len(self.alternatives)
        m = len(self.scores.columns)

        # Normalize data
        normalized = normalize_minmax(self.scores, self.directions)

        # Calculate pairwise comparisons
        phi_plus = pd.Series(0, index=self.alternatives)
        phi_minus = pd.Series(0, index=self.alternatives)

        for i in range(n):
            for j in range(n):
                if i != j:
                    total_pref = 0
                    for k in range(m):
                        criterion = self.scores.columns[k]
                        diff = normalized.iloc[i, k] - normalized.iloc[j, k]

                        # Use .get() for safety and specify direction if not found (though it should be)
                        direction = self.directions.get(criterion)
                        if direction == 'min':
                            diff = -diff
                        elif direction is None: # Handle cases where direction might be missing
                             # This case should ideally not be reached if all columns have directions.
                             # For now, we can skip or assume a default behavior (e.g., maximize)
                             pass

                        pref = self.preference_function(diff)
                        # Ensure weight is retrieved correctly using .get()
                        total_pref += self.weights.get(criterion, 0) * pref

                    if total_pref > 0:
                        phi_plus.iloc[i] += total_pref / (n - 1)
                        phi_minus.iloc[j] += total_pref / (n - 1)

        # Calculate net outranking flow
        phi_net = phi_plus - phi_minus

        results = pd.DataFrame({
            'Phi_Plus': phi_plus,
            'Phi_Minus': phi_minus,
            'Phi_Net': phi_net,
            'Rank': phi_net.rank(ascending=False)
        })

        return results.sort_values('Phi_Net', ascending=False)

# Run PROMETHEE
promethee = PROMETHEE(scores_df, aligned_weights, aligned_directions)
promethee_results = promethee.calculate()

print("\n" + "="*60)
print("PROMETHEE RESULTS")
print("="*60)
print(promethee_results.round(4))


PROMETHEE RESULTS
            Phi_Plus  Phi_Minus  Phi_Net  Rank
London        0.3779     0.2269   0.1511   1.0
Manchester    0.1989     0.1334   0.0655   2.0
Bristol       0.1788     0.1322   0.0467   3.0
Birmingham    0.2014     0.2543  -0.0529   4.0
Cambridge     0.2018     0.2628  -0.0610   5.0
Edinburgh     0.1266     0.2760  -0.1494   6.0


In [ ]:
def sensitivity_analysis(scores, weights, directions, criterion_to_vary, weight_range=None):
    """Perform one-way sensitivity analysis"""
    if weight_range is None:
        weight_range = np.linspace(0.05, 0.40, 20)

    results = []
    # Use .get() with a default or ensure the criterion_to_vary exists in weights
    # For sensitivity analysis, it should exist, otherwise it's an invalid criterion
    base_weight = weights.get(criterion_to_vary)
    if base_weight is None:
        raise ValueError(f"Criterion '{criterion_to_vary}' not found in provided weights.")

    other_weights = {k: v for k, v in weights.items() if k != criterion_to_vary}
    total_other = sum(other_weights.values())

    for new_weight in weight_range:
        # Adjust other weights proportionally
        # Ensure the sum of other weights doesn't become 0 if we're varying a substantial weight
        if total_other > 0:
            adjustment_factor = (1 - new_weight) / total_other
        else: # If all other weights are 0, this criterion takes the full 1.0 weight
            adjustment_factor = 0 # Or handle as an edge case where this is the only criterion

        adjusted_weights = {k: v * adjustment_factor for k, v in other_weights.items()}
        adjusted_weights[criterion_to_vary] = new_weight

        # Ensure weights sum to 1, or close to 1 due to floating point arithmetic
        # print(f"Adjusted weights sum: {sum(adjusted_weights.values())}")

        # Calculate scores
        wsm = WeightedSumModel(scores, adjusted_weights, directions)
        result = wsm.calculate(normalization='minmax')

        # Get ranking
        ranking = result['Total_Score'].rank(ascending=False)
        top_location = ranking.idxmin() # idxmin gives the index of the minimum rank (i.e., rank 1)

        results.append({
            'Weight': new_weight,
            'Top_Location': top_location,
            'Score': result.loc[top_location, 'Total_Score']
        })

    return pd.DataFrame(results)

# Example: Vary the weight of 'Monthly Rent'
# Use aligned_weights and aligned_directions
rent_sensitivity = sensitivity_analysis(
    scores_df, aligned_weights, aligned_directions,
    'Monthly Rent', np.linspace(0.05, 0.50, 15)
)

print("\n" + "="*60)
print("SENSITIVITY ANALYSIS: Monthly Rent Weight")
print("="*60)
print(rent_sensitivity)


SENSITIVITY ANALYSIS: Monthly Rent Weight
      Weight Top_Location     Score
0   0.050000       London  0.935798
1   0.082143       London  0.937970
2   0.114286       London  0.940142
3   0.146429       London  0.942314
4   0.178571       London  0.944487
5   0.210714       London  0.946659
6   0.242857       London  0.948831
7   0.275000       London  0.951003
8   0.307143       London  0.953176
9   0.339286       London  0.955348
10  0.371429       London  0.957520
11  0.403571       London  0.959693
12  0.435714       London  0.961865
13  0.467857       London  0.964037
14  0.500000       London  0.966209


In [ ]:
def scenario_analysis(scores, base_weights, directions):
    """Analyze different decision scenarios"""
    scenarios = {
        'Career_Focused': {
            'Employment': 0.40,
            'Accommodation': 0.20,
            'Transport': 0.15,
            'Financial Costs': 0.15,
            'Personal Preferences': 0.05,
            'External Risks': 0.05
        },
        'Cost_Conscious': {
            'Employment': 0.15,
            'Accommodation': 0.35,
            'Transport': 0.15,
            'Financial Costs': 0.25,
            'Personal Preferences': 0.05,
            'External Risks': 0.05
        },
        'Lifestyle_Focused': {
            'Employment': 0.15,
            'Accommodation': 0.20,
            'Transport': 0.15,
            'Financial Costs': 0.10,
            'Personal Preferences': 0.30,
            'External Risks': 0.10
        },
        'Risk_Averse': {
            'Employment': 0.20,
            'Accommodation': 0.25,
            'Transport': 0.15,
            'Financial Costs': 0.15,
            'Personal Preferences': 0.10,
            'External Risks': 0.15
        }
    }

    scenario_results = {}

    for scenario_name, scenario_weights in scenarios.items():
        # Map scenario weights to individual criteria
        adjusted_weights = {}
        for criterion in base_weights.keys():
            # Find which category this criterion belongs to
            for category in criteria_hierarchy.keys():
                if criterion.startswith(category) or category in criterion:
                    if category in scenario_weights:
                        # Calculate proportional weight within category
                        category_weight = base_weights[criterion] / sum(
                            v for k, v in base_weights.items()
                            if k.startswith(category) or category in k
                        )
                        adjusted_weights[criterion] = scenario_weights[category] * category_weight
                        break

        # Normalize weights
        total = sum(adjusted_weights.values())
        adjusted_weights = {k: v/total for k, v in adjusted_weights.items()}

        # Calculate scores
        wsm = WeightedSumModel(scores, adjusted_weights, directions)
        result = wsm.calculate(normalization='minmax')
        scenario_results[scenario_name] = result['Total_Score']

    return pd.DataFrame(scenario_results)

# Run scenario analysis
scenario_results = scenario_analysis(scores_df, weight_map, direction_map)

print("\n" + "="*60)
print("SCENARIO ANALYSIS RESULTS")
print("="*60)
print(scenario_results.round(3))


SCENARIO ANALYSIS RESULTS
            Career_Focused  Cost_Conscious  Lifestyle_Focused  Risk_Averse
London                   0               0                  0            0
Manchester               0               0                  0            0
Bristol                  0               0                  0            0
Edinburgh                0               0                  0            0
Birmingham               0               0                  0            0
Cambridge                0               0                  0            0


In [ ]:
def visualize_results(wsm_results, topsis_results, promethee_results, scenario_results):
    """Create comprehensive visualizations"""

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=('Weighted Sum Model', 'TOPSIS', 'PROMETHEE',
                        'Scenario Analysis', 'Heatmap', 'Radar Chart'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}],
               [{'type': 'bar'}, {'type': 'heatmap'}, {'type': 'scatterpolar'}]]
    )

    # 1. Weighted Sum Model Results
    fig.add_trace(
        go.Bar(x=wsm_results.index, y=wsm_results['Total_Score'],
               name='WSM', marker_color='royalblue'),
        row=1, col=1
    )

    # 2. TOPSIS Results
    fig.add_trace(
        go.Bar(x=topsis_results.index, y=topsis_results['Closeness'],
               name='TOPSIS', marker_color='crimson'),
        row=1, col=2
    )

    # 3. PROMETHEE Results
    fig.add_trace(
        go.Bar(x=promethee_results.index, y=promethee_results['Phi_Net'],
               name='PROMETHEE', marker_color='forestgreen'),
        row=1, col=3
    )

    # 4. Scenario Analysis
    for scenario in scenario_results.columns:
        fig.add_trace(
            go.Bar(x=scenario_results.index, y=scenario_results[scenario],
                   name=scenario),
            row=2, col=1
        )

    # 5. Heatmap of all methods
    combined = pd.DataFrame({
        'WSM': wsm_results['Total_Score'],
        'TOPSIS': topsis_results['Closeness'],
        'PROMETHEE': promethee_results['Phi_Net']
    })
    fig.add_trace(
        go.Heatmap(z=combined.values, x=combined.columns, y=combined.index,
                   colorscale='RdBu', text=combined.values.round(2), texttemplate='%{text}'),
        row=2, col=2
    )

    # 6. Radar Chart - Top 3 criteria for each location
    top_criteria = scores_df.columns[:6]  # Top 6 criteria
    for location in scores_df.index:
        fig.add_trace(
            go.Scatterpolar(r=scores_df.loc[location, top_criteria].values,
                           theta=top_criteria,
                           name=location,
                           fill='toself'),
            row=2, col=3
        )

    fig.update_layout(height=800, width=1200, showlegend=True)
    fig.show()

# Visualize all results
visualize_results(wsm_results, topsis_results, promethee_results, scenario_results)

In [ ]:
def generate_report(wsm_results, topsis_results, promethee_results, scenario_results):
    """Generate comprehensive decision report"""

    print("\n" + "="*80)
    print("HOUSING LOCATION DECISION ANALYSIS - FINAL REPORT")
    print("="*80)

    # Overall ranking (average of all methods)
    methods = ['WSM', 'TOPSIS', 'PROMETHEE']
    rankings = pd.DataFrame({
        'WSM': wsm_results['Total_Score'],
        'TOPSIS': topsis_results['Closeness'],
        'PROMETHEE': promethee_results['Phi_Net']
    })

    # Normalize each method's scores for fair averaging
    normalized_rankings = (rankings - rankings.min()) / (rankings.max() - rankings.min())
    normalized_rankings['Average'] = normalized_rankings.mean(axis=1)
    normalized_rankings['Overall_Rank'] = normalized_rankings['Average'].rank(ascending=False)

    print("\nOVERALL RANKING (Average of all methods):")
    print(normalized_rankings[['Average', 'Overall_Rank']].sort_values('Overall_Rank').round(3))

    # Best location per scenario
    print("\nBEST LOCATION PER SCENARIO:")
    for scenario in scenario_results.columns:
        best = scenario_results[scenario].idxmax()
        score = scenario_results[scenario].max()
        print(f"  {scenario}: {best} (Score: {score:.3f})")

    # Recommendation
    overall_best = normalized_rankings['Average'].idxmax()
    overall_score = normalized_rankings.loc[overall_best, 'Average']

    print(f"\n" + "="*80)
    print(f"RECOMMENDATION: {overall_best} is the top choice with an overall score of {overall_score:.3f}")
    print("="*80)

    # Detailed strengths and weaknesses
    print("\nSTRENGTHS AND WEAKNESSES:")
    print("-"*40)
    for location in scores_df.index:
        scores = scores_df.loc[location]
        max_criteria = scores.nlargest(3)
        min_criteria = scores.nsmallest(3)
        print(f"\n{location}:")
        print(f"  Top strengths: {', '.join([f'{c} ({s:.1f})' for c, s in max_criteria.items()])}")
        print(f"  Weaknesses: {', '.join([f'{c} ({s:.1f})' for c, s in min_criteria.items()])}")

# Generate final report
generate_report(wsm_results, topsis_results, promethee_results, scenario_results)


HOUSING LOCATION DECISION ANALYSIS - FINAL REPORT

OVERALL RANKING (Average of all methods):
            Average  Overall_Rank
London        1.000           1.0
Bristol       0.381           2.0
Manchester    0.356           3.0
Cambridge     0.353           4.0
Birmingham    0.107           5.0
Edinburgh     0.083           6.0

BEST LOCATION PER SCENARIO:
  Career_Focused: London (Score: 0.000)
  Cost_Conscious: London (Score: 0.000)
  Lifestyle_Focused: London (Score: 0.000)
  Risk_Averse: London (Score: 0.000)

RECOMMENDATION: London is the top choice with an overall score of 1.000

STRENGTHS AND WEAKNESSES:
----------------------------------------

London:
  Top strengths: Job Availability (10.0), Salary Potential (10.0), Availability (10.0)
  Weaknesses: Living Costs (2.0), Monthly Rent (3.0), Deposit Required (4.0)

Manchester:
  Top strengths: Remote Work Flex (9.0), Job Availability (8.0), Monthly Rent (8.0)
  Weaknesses: Savings Potential (4.0), Climate (4.0), Net Income (5.

In [ ]:
def run_mcda_complete():
    """Run complete MCDA analysis"""

    print("="*80)
    print("MULTI-CRITERIA DECISION ANALYSIS - HOUSING LOCATION")
    print("="*80)

    # 1. Input data
    print("\n[1] Loading data...")
    # Data is already defined above

    # 2. Calculate weights
    print("\n[2] Calculating weights...")
    # Weights already calculated above

    # 3. Run Weighted Sum Model
    print("\n[3] Running Weighted Sum Model...")
    print(wsm_results[['Total_Score']])

    # 4. Run TOPSIS
    print("\n[4] Running TOPSIS...")
    print(topsis_results[['Closeness', 'Rank']])

    # 5. Run PROMETHEE
    print("\n[5] Running PROMETHEE...")
    print(promethee_results[['Phi_Net', 'Rank']])

    # 6. Run Sensitivity Analysis
    print("\n[6] Running Sensitivity Analysis...")
    print("Varying 'Monthly Rent' weight:")
    print(rent_sensitivity)

    # 7. Run Scenario Analysis
    print("\n[7] Running Scenario Analysis...")
    print(scenario_results)

    # 8. Generate visualizations
    print("\n[8] Generating visualizations...")
    visualize_results(wsm_results, topsis_results, promethee_results, scenario_results)

    # 9. Generate final report
    print("\n[9] Generating final report...")
    generate_report(wsm_results, topsis_results, promethee_results, scenario_results)

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE!")
    print("="*80)

# Run the complete analysis
if __name__ == "__main__":
    run_mcda_complete()

MULTI-CRITERIA DECISION ANALYSIS - HOUSING LOCATION

[1] Loading data...

[2] Calculating weights...

[3] Running Weighted Sum Model...
Criterion   Total_Score
London         1.152350
Cambridge      0.632550
Bristol        0.547810
Manchester     0.466105
Edinburgh      0.410798
Birmingham     0.266186

[4] Running TOPSIS...
Criterion   Closeness  Rank
London       0.926415   1.0
Cambridge    0.516189   2.0
Bristol      0.402853   3.0
Manchester   0.373614   4.0
Edinburgh    0.347335   5.0
Birmingham   0.293784   6.0

[5] Running PROMETHEE...
             Phi_Net  Rank
London      0.151073   1.0
Manchester  0.065523   2.0
Bristol     0.046669   3.0
Birmingham -0.052856   4.0
Cambridge  -0.061015   5.0
Edinburgh  -0.149394   6.0

[6] Running Sensitivity Analysis...
Varying 'Monthly Rent' weight:
      Weight Top_Location     Score
0   0.050000       London  0.935798
1   0.082143       London  0.937970
2   0.114286       London  0.940142
3   0.146429       London  0.942314
4   0.178571  


[9] Generating final report...

HOUSING LOCATION DECISION ANALYSIS - FINAL REPORT

OVERALL RANKING (Average of all methods):
            Average  Overall_Rank
London        1.000           1.0
Bristol       0.381           2.0
Manchester    0.356           3.0
Cambridge     0.353           4.0
Birmingham    0.107           5.0
Edinburgh     0.083           6.0

BEST LOCATION PER SCENARIO:
  Career_Focused: London (Score: 0.000)
  Cost_Conscious: London (Score: 0.000)
  Lifestyle_Focused: London (Score: 0.000)
  Risk_Averse: London (Score: 0.000)

RECOMMENDATION: London is the top choice with an overall score of 1.000

STRENGTHS AND WEAKNESSES:
----------------------------------------

London:
  Top strengths: Job Availability (10.0), Salary Potential (10.0), Availability (10.0)
  Weaknesses: Living Costs (2.0), Monthly Rent (3.0), Deposit Required (4.0)

Manchester:
  Top strengths: Remote Work Flex (9.0), Job Availability (8.0), Monthly Rent (8.0)
  Weaknesses: Savings Potential (4.0

In [ ]:

import numpy as np
import pandas as pd


def calculate_entropy_weights(df, cost_criteria=None):
    """Calculates criteria weights using the Entropy Weighting Method (EWM).

    Parameters:
    -----------
    df : pd.DataFrame
        Decision matrix (rows = alternatives/suppliers, columns = criteria).
    cost_criteria : list, optional
        List of column names where lower values are better (e.g., Cost, Risk).
        All other columns are treated as benefit criteria (higher is better).

    Returns:
    --------
    weights : pd.Series
        Normalized entropy weights for each criterion (sums to 1.0).
    """
    if cost_criteria is None:
        cost_criteria = []

    matrix = df.astype(float).copy()
    m, n = matrix.shape

    # Step 1: Min-Max Normalization (scale to [0, 1])
    normalized_df = pd.DataFrame(index=df.index, columns=df.columns)

    for col in matrix.columns:
        min_val = matrix[col].min()
        max_val = matrix[col].max()

        if max_val == min_val:
            normalized_df[col] = 1.0 / m
            continue

        if col in cost_criteria:
            # Cost criterion: lower is better
            normalized_df[col] = (max_val - matrix[col]) / (max_val - min_val)
        else:
            # Benefit criterion: higher is better
            normalized_df[col] = (matrix[col] - min_val) / (max_val - min_val)

    # Step 2: Calculate proportion p_ij
    col_sums = normalized_df.sum(axis=0).replace(0, 1.0)
    p_ij = normalized_df.div(col_sums, axis=1)

    # Step 3: Compute Entropy value (E_j) for each criterion
    # k = 1 / ln(m) where m is the number of alternatives
    k = 1.0 / np.log(m)

    # p * ln(p), replacing 0s with 0 to prevent log(0) NaN errors
    p_log_p = p_ij.applymap(lambda v: v * np.log(v) if v > 0 else 0.0)
    entropy = -k * p_log_p.sum(axis=0)

    # Step 4: Calculate Degree of Diversification (d_j)
    d_j = 1.0 - entropy

    # Step 5: Normalize to get final weights (w_j)
    weights = d_j / d_j.sum()

    return weights


# ==========================================
# Example Usage: Farm Food Supplier Selection
# ==========================================

data = {
    "Cost_per_Ton": [1200, 950, 1500, 1100],  # Cost criterion (Lower is better)
    "Shelf_Life_Days": [12, 7, 14, 10],  # Benefit criterion (Higher is better)
    "Sustainability": [4, 3, 5, 2],  # Benefit criterion (Higher is better)
}

df_suppliers = pd.DataFrame(
    data, index=["Farm A", "Farm B", "Farm C", "Farm D"]
)

# Run EWM
weights = calculate_entropy_weights(
    df_suppliers, cost_criteria=["Cost_per_Ton"]
)

print("--- Decision Matrix ---")
print(df_suppliers)
print("\n--- Calculated Entropy Weights ---")
print(weights.round(4))